# A. Setup


In [ ]:
import gc, json, random, tempfile, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
SEED = 42
MAX_EPOCHS_PER_PHASE = 50
EARLY_STOPPING_PATIENCE = 5
IMAGE_SIZE = (224, 224)
HIDDEN_DIM = 512
NUM_CLASSES = 4
OPTIMIZERS = ('Adam', 'AdamW')
LEARNING_RATES = (0.001, 0.0003, 0.0001)
DROPOUT_RATES = (0.4, 0.5)
FINETUNE_LEVELS = ('frozen', 'medium', 'deep')
CLASS_NAMES = ('leaf_curl', 'leaf_spot', 'yellowish', 'healthy_leaf')
CLASS_DISPLAY_NAMES = ('Leaf Curl', 'Leaf Spot', 'Yellowish', 'Healthy Leaf')
MODEL_COLORS = {'mobilenet': '#1f77b4', 'efficientnet': '#ff7f0e', 'hybrid': '#2ca02c'}
DATA_ROOT = Path('data/processed')
TRAIN_MANIFEST = Path('data/splits/train.csv')
VALIDATION_MANIFEST = Path('data/splits/validation.csv')
CHECKPOINT_DIR = Path('checkpoints')
REPORT_DIR = Path('reports')
SCREENING_PATH = REPORT_DIR / 'screening_results.csv'
HISTORY_DIR = REPORT_DIR / 'histories'
CONFUSION_DIR = REPORT_DIR / 'confusion'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('CUDA is required to run the experiment notebook.')

# B. Data


In [ ]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1), fill=0),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
VALIDATION_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

def set_class_order(dataset):
    old_indices = {name: i for i, name in enumerate(dataset.classes)}
    old_to_new = {old_indices[name]: i for i, name in enumerate(CLASS_NAMES)}
    dataset.classes = list(CLASS_NAMES)
    dataset.class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
    dataset.samples = [(path, old_to_new[target]) for path, target in dataset.samples]
    dataset.imgs = dataset.samples
    dataset.targets = [target for _, target in dataset.samples]
    return dataset
train_dataset = set_class_order(datasets.ImageFolder(
    DATA_ROOT / 'train',
    transform=TRAIN_TRANSFORM
))
validation_dataset = set_class_order(datasets.ImageFolder(
    DATA_ROOT / 'validation',
    transform=VALIDATION_TRANSFORM
))
assert train_dataset.classes == validation_dataset.classes == list(CLASS_NAMES)
count_rows = []
for split, path in (
    ('train', TRAIN_MANIFEST),
    ('validation', VALIDATION_MANIFEST),
):
    counts = pd.read_csv(path)['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    count_rows.append({'Split': split, **counts.to_dict()})
display(pd.DataFrame(count_rows).set_index('Split'))

,leaf_curl,leaf_spot,yellowish,healthy_leaf
Split,,,,
train,257,281,267,281
validation,55,59,58,61


## 1. Reproducibility and device


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator

## 2. Data

In [ ]:
def make_evaluation_loader(dataset, batch_size):
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

def build_dataloaders(batch_size, seed):
    generator = seed_everything(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True
    )
    return (train_loader, make_evaluation_loader(validation_dataset, batch_size))

## 3. Models


In [ ]:
def build_head(input_features, dropout):
    return nn.Sequential(
        nn.Linear(input_features, HIDDEN_DIM),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(HIDDEN_DIM, NUM_CLASSES)
    )

class FeatureBackbone(nn.Module):

    def __init__(self, features):
        super().__init__()
        self.features = features
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, images):
        return self.pool(self.features(images)).flatten(1)

class SingleBackboneModel(nn.Module):

    def __init__(self, features, dropout):
        super().__init__()
        self.features = features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = build_head(1280, dropout)

    def forward(self, images):
        return self.head(self.pool(self.features(images)).flatten(1))

class FusionModel(nn.Module):

    def __init__(self, mobile_features, efficient_features, dropout):
        super().__init__()
        self.mobilenet = FeatureBackbone(mobile_features)
        self.efficientnet = FeatureBackbone(efficient_features)
        self.head = build_head(2560, dropout)

    def forward(self, images):
        return self.head(torch.cat((self.mobilenet(images), self.efficientnet(images)), dim=1))

def build_base_model(model_name, dropout):
    if model_name == 'mobilenet':
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    elif model_name == 'efficientnet':
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    else:
        raise ValueError(f'Unsupported model: {model_name}')
    return SingleBackboneModel(backbone.features, dropout).to(DEVICE)

def build_hybrid_model(dropout):
    mobile = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    efficient = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    return FusionModel(mobile.features, efficient.features, dropout).to(DEVICE)

def count_parameters(model):
    return sum((parameter.numel() for parameter in model.parameters()))

## 4. Fine-tuning Control


In [ ]:
def apply_finetune_level(model, finetune_level):
    if not isinstance(model, FusionModel):
        for parameter in model.parameters():
            parameter.requires_grad = True
        return
    if finetune_level not in FINETUNE_LEVELS:
        raise ValueError('Hybrid scenarios require a fine-tuning level.')
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.head.parameters():
        parameter.requires_grad = True
    if finetune_level == 'frozen':
        return
    ranges = {'medium': ((14, 19), (6, 9)), 'deep': ((10, 19), (4, 9))}
    for backbone, (start, stop) in zip(
        (model.mobilenet, model.efficientnet),
        ranges[finetune_level]
    ):
        for block in backbone.features[start:stop]:
            for parameter in block.parameters():
                parameter.requires_grad = True

def set_training_mode(model, phase):
    model.train()
    if isinstance(model, FusionModel):
        if phase == 1:
            model.mobilenet.eval()
            model.efficientnet.eval()
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
    return model

def build_optimizer(model, optimizer_name, learning_rate):
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if optimizer_name == 'Adam':
        return torch.optim.Adam(parameters, lr=learning_rate, weight_decay=0.0)
    if optimizer_name == 'AdamW':
        return torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=0.01)
    raise ValueError(f'Unsupported optimizer: {optimizer_name}')

## 5. Training

In [ ]:
def train_one_phase(
    model,
    train_loader,
    validation_loader,
    optimizer,
    scaler,
    phase,
    show_epoch_log
):
    criterion = nn.CrossEntropyLoss()
    best_loss = float('inf')
    best_state = None
    best_epoch = 0
    wait = 0
    history = []
    for epoch in range(1, MAX_EPOCHS_PER_PHASE + 1):
        set_training_mode(model, phase)
        started = time.perf_counter()
        loss_sum = 0.0
        correct = 0
        count = 0
        for images, targets in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(targets)
            correct += (logits.argmax(1) == targets).sum().item()
            count += len(targets)
        if not count:
            raise ValueError('Training data loader is empty.')
        train_loss = loss_sum / count
        train_accuracy = correct / count
        targets_all, predictions, _, val_loss = evaluate_model(model, validation_loader)
        val_accuracy = accuracy_score(targets_all, predictions)
        seconds = time.perf_counter() - started
        history.append({
            'phase': phase,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_accuracy': train_accuracy,
            'validation_loss': val_loss,
            'validation_accuracy': val_accuracy,
            'epoch_seconds': seconds
        })
        if show_epoch_log:
            print(
                f"phase {phase} | epoch {epoch:02d} | train loss {train_loss:.4f} "
                f"acc {train_accuracy:.4f} | val loss {val_loss:.4f} "
                f"acc {val_accuracy:.4f} | {seconds:.1f} s"
            )
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            wait = 0
            best_state = {
                key: value.detach().cpu().clone() for key,
                value in model.state_dict().items()
            }
        else:
            wait += 1
        if wait >= EARLY_STOPPING_PATIENCE:
            break
    if best_state is None:
        raise RuntimeError('Training did not produce a best state.')
    model.load_state_dict(best_state)
    if show_epoch_log:
        print(f'phase {phase} stopped at epoch {len(history)}, best epoch {best_epoch}')
    return (model, history, best_epoch)

def train_scenario(
    scenario_id,
    model_name,
    optimizer_name,
    learning_rate,
    dropout,
    batch_size,
    finetune_level=None,
    seed=42,
    show_epoch_log=False
):
    seed_everything(seed)
    train_loader, val_loader = build_dataloaders(batch_size, seed)
    model = build_hybrid_model(dropout) if model_name == 'hybrid' else build_base_model(
        model_name,
        dropout
    )
    if model_name == 'hybrid':
        apply_finetune_level(model, 'frozen')
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    started = time.perf_counter()
    optimizer = build_optimizer(model, optimizer_name, learning_rate)
    model, phase_one_history, _ = train_one_phase(
        model,
        train_loader,
        val_loader,
        optimizer,
        scaler,
        1,
        show_epoch_log
    )
    history = phase_one_history
    epochs = [len(phase_one_history)]
    if model_name == 'hybrid' and finetune_level != 'frozen':
        apply_finetune_level(model, finetune_level)
        optimizer = build_optimizer(model, optimizer_name, learning_rate)
        model, phase_two_history, _ = train_one_phase(
            model,
            train_loader,
            val_loader,
            optimizer,
            scaler,
            2,
            show_epoch_log
        )
        history = phase_one_history + phase_two_history
        epochs.append(len(phase_two_history))
    checkpoint = CHECKPOINT_DIR / f'{scenario_id}_seed{seed}.pt'
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, checkpoint)
    seconds = time.perf_counter() - started
    del train_loader, val_loader
    return (model, history, epochs, seconds)

## 6. Evaluation


In [ ]:
def evaluate_model(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum = 0.0
    targets = []
    predictions = []
    probabilities = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels_device = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            loss_sum += criterion(logits, labels_device).item() * len(labels)
            targets.extend(labels.numpy().tolist())
            predictions.extend(logits.argmax(1).cpu().numpy().tolist())
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
    if not targets:
        raise ValueError('Evaluation data loader is empty.')
    return (
        np.asarray(targets),
        np.asarray(predictions),
        np.concatenate(probabilities),
        loss_sum / len(targets)
    )

def compute_metrics(targets, predictions, probabilities, loss):
    return {
        'Accuracy': accuracy_score(targets, predictions),
        'Precision': precision_score(targets, predictions, average='macro', zero_division=0),
        'Recall': recall_score(targets, predictions, average='macro', zero_division=0),
        'F1': f1_score(targets, predictions, average='macro', zero_division=0),
        'Loss': float(loss),
        'AUC': roc_auc_score(
            targets,
            probabilities,
            labels=np.arange(NUM_CLASSES),
            average='macro',
            multi_class='ovr'
        )
    }

def compute_per_class_metrics(true_labels, predicted_labels):
    metric_rows = []
    for class_index, class_name in enumerate(CLASS_DISPLAY_NAMES):
        metric_rows.append({
            'Class': class_name,
            'Precision': precision_score(
                true_labels,
                predicted_labels,
                labels=[class_index],
                average='macro',
                zero_division=0
            ),
            'Recall': recall_score(
                true_labels,
                predicted_labels,
                labels=[class_index],
                average='macro',
                zero_division=0
            ),
            'F1': f1_score(
                true_labels,
                predicted_labels,
                labels=[class_index],
                average='macro',
                zero_division=0
            )
        })
    return pd.DataFrame(metric_rows)

## 7. Efficiency & Storage


In [ ]:
def measure_efficiency(model):
    params = count_parameters(model)
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as file:
        path = Path(file.name)
    try:
        torch.save(model.state_dict(), path)
        size = path.stat().st_size / 1024 ** 2
    finally:
        path.unlink(missing_ok=True)
    model.eval()
    sample = torch.zeros(1, 3, *IMAGE_SIZE, device=DEVICE)
    timings = []
    with torch.no_grad():
        for _ in range(20):
            model(sample)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        for _ in range(100):
            start = time.perf_counter()
            model(sample)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            timings.append((time.perf_counter() - start) * 1000)
    return {'Params': params, 'Size (MB)': size, 'Latency (ms/img)': float(np.mean(timings))}

def format_metrics_table(metrics, efficiency, epochs, seconds):
    columns = [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'Loss',
        'AUC',
        'Params',
        'Size (MB)',
        'Latency (ms/img)',
        'Epochs',
        'Train time (s)'
    ]
    return pd.DataFrame(
        [{**metrics, **efficiency, 'Epochs': epochs, 'Train time (s)': seconds}],
        columns=columns
    ).round(4)

def create_confusion_heatmap(scenario_id, confusion):
    figure, axis = plt.subplots(figsize=(8, 6))
    image = axis.imshow(confusion, cmap="Blues")
    figure.colorbar(image, ax=axis)
    axis.set(
        xticks=np.arange(NUM_CLASSES),
        yticks=np.arange(NUM_CLASSES),
        xticklabels=CLASS_DISPLAY_NAMES,
        yticklabels=CLASS_DISPLAY_NAMES,
        xlabel="Kelas Prediksi",
        ylabel="Kelas Sebenarnya",
        title=f"Matriks Konfusi Data Validasi: {scenario_id}",
    )
    threshold = confusion.max() / 2
    for row_index in range(NUM_CLASSES):
        for column_index in range(NUM_CLASSES):
            value = confusion[row_index, column_index]
            color = "white" if value > threshold else "black"
            axis.text(
                column_index,
                row_index,
                str(value),
                ha="center",
                va="center",
                color=color,
            )
    figure.tight_layout()
    return figure

def save_result(scenario_row, history, confusion):
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    CONFUSION_DIR.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame([scenario_row])
    if SCREENING_PATH.exists():
        previous = pd.read_csv(SCREENING_PATH)
        frame = pd.concat(
            [previous[previous.scenario_id != scenario_row['scenario_id']], frame],
            ignore_index=True
        )
    frame.to_csv(SCREENING_PATH, index=False)
    pd.DataFrame(history).to_csv(HISTORY_DIR / f"{scenario_row['scenario_id']}.csv", index=False)
    figure = create_confusion_heatmap(scenario_row['scenario_id'], confusion)
    figure.savefig(CONFUSION_DIR / f"{scenario_row['scenario_id']}.png", dpi=300)
    plt.close(figure)

def show_result_tables(
    scenario_id,
    metrics=None,
    efficiency=None,
    epochs=0,
    seconds=0.0,
    confusion=None
):
    if metrics is None:
        scenario_row = pd.read_csv(SCREENING_PATH).query('scenario_id == @scenario_id').iloc[0]
        cols = [
            'Accuracy',
            'Precision',
            'Recall',
            'F1',
            'Loss',
            'AUC',
            'Params',
            'Size (MB)',
            'Latency (ms/img)',
            'Epochs',
            'Train time (s)'
        ]
        display(pd.DataFrame([scenario_row[cols].to_dict()]).round(4))
        display(Image(filename=CONFUSION_DIR / f'{scenario_id}.png'))
    else:
        display(format_metrics_table(metrics, efficiency, epochs, seconds))
        figure = create_confusion_heatmap(scenario_id, confusion)
        display(figure)
        plt.close(figure)

def show_progress():
    results = pd.read_csv(SCREENING_PATH) if SCREENING_PATH.exists() else pd.DataFrame()
    variants = (
        ('MobileNetV2', 'mobilenet', None),
        ('EfficientNetB0', 'efficientnet', None),
        ('Hybrid frozen', 'hybrid', 'frozen'),
        ('Hybrid medium', 'hybrid', 'medium'),
        ('Hybrid deep', 'hybrid', 'deep')
    )
    rows = []
    for title, name, level in variants:
        subset = results[results.status == 'ok'] if not results.empty else results
        if not subset.empty:
            subset = subset[subset.model == name]
            if level:
                subset = subset[subset.finetune_level == level]
        rows.append({'Model variant': title, 'Scenarios done': len(subset), 'Scenarios total': 12})
    display(pd.DataFrame(rows))

def discard_scenario_results(scenario_ids):
    backup_path = REPORT_DIR / f"backup_{time.strftime('%Y%m%d_%H%M')}"
    backup_path.mkdir(parents=True, exist_ok=True)
    csv_paths = list(REPORT_DIR.glob('*.csv'))
    for csv_path in csv_paths:
        csv_path.replace(backup_path / csv_path.name)
    result_files = ('screening_results.csv', 'seed_results.csv', 'test_results.csv')
    for file_name in result_files:
        backup_csv = backup_path / file_name
        if not backup_csv.exists():
            continue
        stored_results = pd.read_csv(backup_csv)
        stored_results = stored_results[~stored_results.scenario_id.isin(scenario_ids)]
        stored_results.to_csv(REPORT_DIR / file_name, index=False)
    for directory in (HISTORY_DIR, CONFUSION_DIR, CHECKPOINT_DIR):
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if any((path.name.startswith(scenario_id) for scenario_id in scenario_ids)):
                if path.is_file():
                    path.unlink()

## 8. Screening


In [ ]:
def run_scenario(
    scenario_id,
    model,
    optimizer,
    lr,
    dropout,
    batch_size,
    finetune_level=None,
    seed=42,
    rerun=False
):
    if SCREENING_PATH.exists() and (not rerun):
        stored_results = pd.read_csv(SCREENING_PATH)
        completed_scenario = stored_results[
            (stored_results.scenario_id == scenario_id)
            & (stored_results.status == "ok")
        ]
        if not completed_scenario.empty:
            show_result_tables(scenario_id)
            return
    started = time.perf_counter()
    instance = None
    history = []
    epochs = []
    metrics = {}
    efficiency = {}
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    status = 'ok'
    try:
        instance, history, epochs, seconds = train_scenario(
            scenario_id,
            model,
            optimizer,
            lr,
            dropout,
            batch_size,
            finetune_level,
            seed,
            False
        )
        loader = make_evaluation_loader(validation_dataset, batch_size)
        true_labels, predicted_labels, class_probabilities, loss = evaluate_model(instance, loader)
        metrics = compute_metrics(true_labels, predicted_labels, class_probabilities, loss)
        efficiency = measure_efficiency(instance)
        cm = confusion_matrix(true_labels, predicted_labels, labels=np.arange(NUM_CLASSES))
        del loader
    except torch.cuda.OutOfMemoryError:
        status = 'oom'
        seconds = time.perf_counter() - started
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    finally:
        if instance is not None:
            del instance
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    scenario_row = {
        'scenario_id': scenario_id,
        'model': model,
        'optimizer': optimizer,
        'lr': lr,
        'dropout': dropout,
        'batch_size': batch_size,
        'finetune_level': finetune_level,
        'seed': seed,
        'status': status,
        **{name: metrics.get(name, np.nan) for name in (
            'Accuracy',
            'Precision',
            'Recall',
            'F1',
            'Loss',
            'AUC'
        )},
        **{name: efficiency.get(name, np.nan) for name in (
            'Params',
            'Size (MB)',
            'Latency (ms/img)'
        )},
        'epochs_per_phase': json.dumps(epochs),
        'Epochs': sum(epochs),
        'Train time (s)': seconds
    }
    save_result(scenario_row, history, cm)
    show_result_tables(scenario_id, metrics or None, efficiency, sum(epochs), seconds, cm)

## 9. Selection

In [ ]:
def rank_scenarios(top_k=2):
    screening = pd.read_csv(SCREENING_PATH)
    screening = screening[screening.status == 'ok'].copy()
    screening['finetune_level'] = screening['finetune_level'].fillna('-')
    unit_columns = ['model', 'finetune_level', 'lr', 'dropout']
    ordered = screening.sort_values(['F1', 'Loss'], ascending=[False, True], kind='stable')
    kept = ordered.drop_duplicates(unit_columns)
    other = ordered.merge(
        kept[unit_columns + ['optimizer']].rename(columns={'optimizer': 'best_optimizer'}),
        on=unit_columns,
        how='inner'
    )
    other = other[other.optimizer != other.best_optimizer]
    other = other[unit_columns + ['F1']].rename(columns={'F1': 'Other optimizer F1'})
    ranked = kept.merge(other, on=unit_columns, how='left')
    names = {'mobilenet': 'MobileNetV2', 'efficientnet': 'EfficientNetB0', 'hybrid': 'Hybrid'}
    for model_name, display_name in names.items():
        model_rows = ranked[ranked.model == model_name].sort_values(
            ['F1', 'Loss'],
            ascending=[False, True],
            kind='stable'
        ).head(top_k).copy()
        model_rows.insert(0, 'Rank', np.arange(1, len(model_rows) + 1))
        model_rows.insert(1, 'Model type', display_name)
        display(model_rows[[
            'Rank',
            'Model type',
            'scenario_id',
            'optimizer',
            'lr',
            'dropout',
            'batch_size',
            'finetune_level',
            'F1',
            'Loss',
            'Other optimizer F1',
            'AUC',
            'Accuracy'
        ]].round(4))

def compare_same_configuration():
    results = pd.read_csv(SCREENING_PATH)
    results = results[results.status == 'ok'].copy()
    results['Model variant'] = results.apply(
        lambda scenario_row: (
            f"Hybrid {scenario_row.finetune_level}"
            if scenario_row.model == "hybrid"
            else "MobileNetV2"
            if scenario_row.model == "mobilenet"
            else "EfficientNetB0"
        ),
        axis=1
    )
    table = results.pivot_table(
        index=['optimizer', 'lr', 'dropout'],
        columns='Model variant',
        values='F1',
        aggfunc='first'
    )
    cols = ['MobileNetV2', 'EfficientNetB0', 'Hybrid frozen', 'Hybrid medium', 'Hybrid deep']
    index = pd.MultiIndex.from_product(
        [OPTIMIZERS, LEARNING_RATES, DROPOUT_RATES],
        names=['optimizer', 'lr', 'dropout']
    )
    display(table.reindex(index=index, columns=cols).reset_index().round(4))

## 10. Seed & Test


In [ ]:
def run_additional_seeds(selected_scenario_ids):
    if not selected_scenario_ids:
        raise ValueError('Selected scenario IDs cannot be empty.')
    screening = pd.read_csv(SCREENING_PATH)
    seed_path = REPORT_DIR / 'seed_results.csv'
    stored_results = pd.read_csv(seed_path) if seed_path.exists() else pd.DataFrame()
    new_results = []
    metric_columns = [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'Loss',
        'AUC',
        'Params',
        'Size (MB)',
        'Latency (ms/img)'
    ]
    for scenario_id in selected_scenario_ids:
        scenario_config = screening[
            (screening.scenario_id == scenario_id)
            & (screening.status == "ok")
        ].iloc[0]
        for seed in (43, 44):
            is_stored = not stored_results.empty and (
                (stored_results.scenario_id == scenario_id)
                & (stored_results.seed == seed)
            ).any()
            if is_stored:
                continue
            model, history, epochs, seconds = train_scenario(
                scenario_id,
                scenario_config.model,
                scenario_config.optimizer,
                float(scenario_config.lr),
                float(scenario_config.dropout),
                int(scenario_config.batch_size),
                scenario_config.finetune_level if scenario_config.model == 'hybrid' else None,
                seed,
                False
            )
            validation_loader = make_evaluation_loader(
                validation_dataset,
                int(scenario_config.batch_size)
            )
            true_labels, predicted_labels, class_probabilities, validation_loss = evaluate_model(
                model,
                validation_loader
            )
            metrics = compute_metrics(
                true_labels,
                predicted_labels,
                class_probabilities,
                validation_loss
            )
            efficiency = measure_efficiency(model)
            new_results.append({'scenario_id': scenario_id, 'seed': seed, **metrics, **efficiency})
            print(
                f"{scenario_id} seed {seed} | validation F1 {metrics['F1']:.4f} "
                f"| {seconds:.0f} s"
            )
            del model, validation_loader, history, epochs
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    if new_results:
        stored_results = pd.concat([stored_results, pd.DataFrame(new_results)], ignore_index=True)
        stored_results.to_csv(seed_path, index=False)
    screening_seeds = screening[
        screening.scenario_id.isin(selected_scenario_ids)
        & (screening.status == "ok")
    ][["scenario_id", "seed", *metric_columns]]
    added_seeds = stored_results[stored_results.scenario_id.isin(selected_scenario_ids)][[
        'scenario_id',
        'seed',
        *metric_columns
    ]]
    all_seeds = pd.concat([screening_seeds, added_seeds], ignore_index=True)
    grouped_metrics = all_seeds.groupby('scenario_id')[metric_columns]
    summary = pd.concat(
        [
            grouped_metrics.mean().add_suffix(' mean'),
            grouped_metrics.std(ddof=1).add_suffix(' std')
        ],
        axis=1
    )
    display(summary.round(4))

def select_final_models():
    screening = pd.read_csv(SCREENING_PATH)
    seed_path = REPORT_DIR / "seed_results.csv"
    additional_seeds = pd.read_csv(seed_path) if seed_path.exists() else pd.DataFrame()
    scenario_ids = SELECTED_SCENARIO_IDS
    screening = screening[
        screening.scenario_id.isin(scenario_ids) & (screening.status == "ok")
    ]
    seed_columns = ["scenario_id", "seed", "F1", "Loss"]
    all_seed_results = screening[seed_columns]
    if not additional_seeds.empty:
        all_seed_results = pd.concat(
            [all_seed_results, additional_seeds[seed_columns]],
            ignore_index=True,
        )
    all_seed_results = all_seed_results.drop_duplicates(
        ["scenario_id", "seed"], keep="first"
    )
    seed_counts = all_seed_results.groupby("scenario_id").seed.nunique()
    complete_scenario_ids = seed_counts[seed_counts == 3].index
    mean_metrics = all_seed_results[
        all_seed_results.scenario_id.isin(complete_scenario_ids)
    ].groupby("scenario_id", as_index=False)[["F1", "Loss"]].mean()
    scenario_details = screening[
        screening.scenario_id.isin(complete_scenario_ids)
    ].drop_duplicates("scenario_id")
    if set(scenario_details.model) != {"mobilenet", "efficientnet", "hybrid"}:
        raise ValueError(
            "Complete validation results for seeds 42, 43, and 44 are required."
        )
    ranked = scenario_details.drop(columns=["F1", "Loss"]).merge(
        mean_metrics,
        on="scenario_id",
    )
    ranked = ranked.sort_values(
        ["model", "F1", "Loss"],
        ascending=[True, False, True],
        kind="stable",
    )
    return ranked.drop_duplicates("model")[[
        "scenario_id",
        "model",
        "optimizer",
        "lr",
        "dropout",
        "batch_size",
        "finetune_level",
        "F1",
    ]].rename(columns={"F1": "Mean validation F1"})

def evaluate_on_test():
    selected_models = select_final_models()
    selected_scenario_ids = selected_models.scenario_id.tolist()
    if not selected_scenario_ids:
        raise ValueError('No selected scenarios are available for final evaluation.')
    test_dataset = set_class_order(datasets.ImageFolder(
        DATA_ROOT / 'test',
        transform=VALIDATION_TRANSFORM
    ))
    result_rows = []
    efficiency_rows = []
    class_rows = []
    confusion_matrices = {}
    for scenario_id in selected_scenario_ids:
        scenario_config = selected_models[selected_models.scenario_id == scenario_id].iloc[0]
        summed_confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
        for seed in (42, 43, 44):
            if scenario_config.model == "hybrid":
                model = build_hybrid_model(float(scenario_config.dropout))
            else:
                model = build_base_model(
                scenario_config.model,
                float(scenario_config.dropout)
            )
            checkpoint_path = CHECKPOINT_DIR / f'{scenario_id}_seed{seed}.pt'
            model.load_state_dict(torch.load(
                checkpoint_path,
                map_location=DEVICE,
                weights_only=True
            ))
            test_loader = make_evaluation_loader(test_dataset, int(scenario_config.batch_size))
            true_labels, predicted_labels, class_probabilities, loss = evaluate_model(
                model,
                test_loader
            )
            metrics = compute_metrics(true_labels, predicted_labels, class_probabilities, loss)
            efficiency = measure_efficiency(model)
            result_rows.append({'scenario_id': scenario_id, 'seed': seed, **metrics, **efficiency})
            efficiency_rows.append({'scenario_id': scenario_id, 'seed': seed, **efficiency})
            summed_confusion += confusion_matrix(
                true_labels,
                predicted_labels,
                labels=np.arange(NUM_CLASSES)
            )
            class_metrics = compute_per_class_metrics(
                true_labels,
                predicted_labels,
            )
            class_metrics.insert(0, 'seed', seed)
            class_metrics.insert(0, 'scenario_id', scenario_id)
            class_rows.extend(class_metrics.to_dict('records'))
            del model, test_loader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        confusion_matrices[scenario_id] = pd.DataFrame(
            summed_confusion,
            index=CLASS_DISPLAY_NAMES,
            columns=CLASS_DISPLAY_NAMES
        )
    results = pd.DataFrame(result_rows)
    results.to_csv(REPORT_DIR / 'test_results.csv', index=False)
    pd.DataFrame(class_rows).to_csv(REPORT_DIR / 'test_per_class.csv', index=False)
    display(results.groupby('scenario_id')[[
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'Loss',
        'AUC'
    ]].agg(['mean', 'std']).round(4))
    display(pd.DataFrame(efficiency_rows).groupby('scenario_id')[[
        'Params',
        'Size (MB)',
        'Latency (ms/img)'
    ]].mean().round(4))
    test_confusion_directory = REPORT_DIR / "test_confusion"
    test_confusion_directory.mkdir(parents=True, exist_ok=True)
    for scenario_id, confusion_matrix_frame in confusion_matrices.items():
        confusion_matrix_frame.to_csv(REPORT_DIR / 'test_confusion' / f'{scenario_id}.csv')
        display(confusion_matrix_frame)
    display(pd.DataFrame(class_rows).groupby(['scenario_id', 'Class'])[[
        'Precision',
        'Recall',
        'F1'
    ]].mean().round(4))
    display(results.merge(
        selected_models[['scenario_id', 'model']],
        on='scenario_id'
    ).groupby('model')[['Accuracy', 'Precision', 'Recall', 'F1', 'Loss', 'AUC']].mean().T.round(4))

## C. Screening

### MobileNetV2

In [ ]:
run_scenario(scenario_id='mobilenet_01', model='mobilenet', optimizer='Adam', lr=0.001, dropout=0.4, batch_size=32)

In [ ]:
run_scenario(
    scenario_id='mobilenet_02',
    model='mobilenet',
    optimizer='Adam',
    lr=0.001,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_03',
    model='mobilenet',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_04',
    model='mobilenet',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_05',
    model='mobilenet',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_06',
    model='mobilenet',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_07',
    model='mobilenet',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_08',
    model='mobilenet',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_09',
    model='mobilenet',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_10',
    model='mobilenet',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_11',
    model='mobilenet',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='mobilenet_12',
    model='mobilenet',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.5,
    batch_size=32
)

### EfficientNetB0

In [ ]:
run_scenario(
    scenario_id='efficientnet_01',
    model='efficientnet',
    optimizer='Adam',
    lr=0.001,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_02',
    model='efficientnet',
    optimizer='Adam',
    lr=0.001,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_03',
    model='efficientnet',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_04',
    model='efficientnet',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_05',
    model='efficientnet',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_06',
    model='efficientnet',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_07',
    model='efficientnet',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_08',
    model='efficientnet',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_09',
    model='efficientnet',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_10',
    model='efficientnet',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.5,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_11',
    model='efficientnet',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.4,
    batch_size=32
)

In [ ]:
run_scenario(
    scenario_id='efficientnet_12',
    model='efficientnet',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.5,
    batch_size=32
)

### Hybrid Freeze

In [ ]:
run_scenario(
    scenario_id='hybrid_01',
    model='hybrid',
    optimizer='Adam',
    lr=0.001,
    dropout=0.4,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_02',
    model='hybrid',
    optimizer='Adam',
    lr=0.001,
    dropout=0.5,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_03',
    model='hybrid',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.4,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_04',
    model='hybrid',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.5,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_05',
    model='hybrid',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.4,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_06',
    model='hybrid',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.5,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_07',
    model='hybrid',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.4,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_08',
    model='hybrid',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.5,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_09',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.4,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_10',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.5,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_11',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.4,
    batch_size=32,
    finetune_level='frozen'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_12',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.5,
    batch_size=32,
    finetune_level='frozen'
)

### Hybrid Medium Unfreeze

In [ ]:
run_scenario(
    scenario_id='hybrid_13',
    model='hybrid',
    optimizer='Adam',
    lr=0.001,
    dropout=0.4,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_14',
    model='hybrid',
    optimizer='Adam',
    lr=0.001,
    dropout=0.5,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_15',
    model='hybrid',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.4,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_16',
    model='hybrid',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.5,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_17',
    model='hybrid',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.4,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_18',
    model='hybrid',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.5,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_19',
    model='hybrid',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.4,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_20',
    model='hybrid',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.5,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_21',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.4,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_22',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.5,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_23',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.4,
    batch_size=32,
    finetune_level='medium'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_24',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.5,
    batch_size=32,
    finetune_level='medium'
)

### Hybrid Deep Unfreeze

In [ ]:
run_scenario(
    scenario_id='hybrid_25',
    model='hybrid',
    optimizer='Adam',
    lr=0.001,
    dropout=0.4,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_26',
    model='hybrid',
    optimizer='Adam',
    lr=0.001,
    dropout=0.5,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_27',
    model='hybrid',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.4,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_28',
    model='hybrid',
    optimizer='Adam',
    lr=0.0003,
    dropout=0.5,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_29',
    model='hybrid',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.4,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_30',
    model='hybrid',
    optimizer='Adam',
    lr=0.0001,
    dropout=0.5,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_31',
    model='hybrid',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.4,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_32',
    model='hybrid',
    optimizer='AdamW',
    lr=0.001,
    dropout=0.5,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_33',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.4,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_34',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0003,
    dropout=0.5,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_35',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.4,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
run_scenario(
    scenario_id='hybrid_36',
    model='hybrid',
    optimizer='AdamW',
    lr=0.0001,
    dropout=0.5,
    batch_size=32,
    finetune_level='deep'
)

In [ ]:
show_progress()

## D. Ranking

In [ ]:
rank_scenarios(top_k=2)

In [ ]:
compare_same_configuration()

## E. Additional Seeds

In [ ]:
SELECTED_SCENARIO_IDS = []
run_additional_seeds(SELECTED_SCENARIO_IDS)

## F. Final Test

In [ ]:
evaluate_on_test()

## G. Visualization

In [ ]:
def get_model_label(scenario_config):
    if scenario_config.model == "mobilenet":
        return "MobileNetV2"
    if scenario_config.model == "efficientnet":
        return "EfficientNetB0"
    return f"Hybrid {scenario_config.finetune_level}"


def prepare_training_history(scenario_id):
    history_path = HISTORY_DIR / f"{scenario_id}.csv"
    if not history_path.exists():
        raise FileNotFoundError(f"Training history not found for {scenario_id}.")
    history = pd.read_csv(history_path)
    phase_one_epochs = int(history.loc[history.phase == 1, "epoch"].max())
    history["cumulative_epoch"] = history.epoch + np.where(
        history.phase == 2, phase_one_epochs, 0
    )
    return history, phase_one_epochs


def plot_training_curves(selected_scenario_ids):
    screening = pd.read_csv(SCREENING_PATH)
    selected_rows = screening[screening.scenario_id.isin(selected_scenario_ids)]
    figure_directory = REPORT_DIR / "figures"
    figure_directory.mkdir(parents=True, exist_ok=True)
    histories = {}
    for scenario_id in selected_scenario_ids:
        scenario_config = selected_rows[
            selected_rows.scenario_id == scenario_id
        ].iloc[0]
        history, phase_one_epochs = prepare_training_history(scenario_id)
        histories[scenario_id] = (history, phase_one_epochs, scenario_config)
        figure, axes = plt.subplots(1, 2, figsize=(15, 5))
        color = MODEL_COLORS[scenario_config.model]
        for axis, train_column, validation_column, ylabel, title, train_label, validation_label in (
            (
                axes[0], "train_loss", "validation_loss", "Loss",
                "Kurva Loss", "Loss Latih", "Loss Validasi",
            ),
            (
                axes[1], "train_accuracy", "validation_accuracy", "Akurasi",
                "Kurva Akurasi", "Akurasi Latih", "Akurasi Validasi",
            ),
        ):
            axis.plot(
                history.cumulative_epoch,
                history[train_column],
                color=color,
                label=train_label,
            )
            axis.plot(
                history.cumulative_epoch,
                history[validation_column],
                color=color,
                linestyle="--",
                label=validation_label,
            )
            axis.set(
                xlabel="Epoch",
                ylabel=ylabel,
                title=f"{title}: {get_model_label(scenario_config)} ({scenario_id})",
            )
            if scenario_config.model == "hybrid" and scenario_config.finetune_level != "frozen":
                axis.axvline(
                    phase_one_epochs + 0.5,
                    color="black",
                    linestyle="--",
                    label="Awal fase 2",
                )
            axis.grid(alpha=0.25)
            axis.legend()
        figure.tight_layout()
        figure.savefig(figure_directory / f"training_curves_{scenario_id}.png", dpi=300)
        display(figure)
        plt.close(figure)
    for model_name in ("mobilenet", "efficientnet", "hybrid"):
        model_ids = selected_rows.loc[
            selected_rows.model == model_name, "scenario_id"
        ].tolist()
        if not model_ids:
            continue
        figure, axes = plt.subplots(
            len(model_ids),
            2,
            figsize=(14, 4 * len(model_ids)),
            squeeze=False,
        )
        for row_index, scenario_id in enumerate(model_ids):
            history, phase_one_epochs, scenario_config = histories[scenario_id]
            for column_index, (train_column, validation_column, ylabel) in enumerate((
                ("train_loss", "validation_loss", "Loss"),
                ("train_accuracy", "validation_accuracy", "Akurasi"),
            )):
                axis = axes[row_index, column_index]
                axis.plot(
                    history.cumulative_epoch,
                    history[train_column],
                    color=MODEL_COLORS[model_name],
                    label="Loss Latih" if column_index == 0 else "Akurasi Latih",
                )
                axis.plot(
                    history.cumulative_epoch,
                    history[validation_column],
                    color=MODEL_COLORS[model_name],
                    linestyle="--",
                    label="Loss Validasi" if column_index == 0 else "Akurasi Validasi",
                )
                axis.set(xlabel="Epoch", ylabel=ylabel, title=scenario_id)
                if model_name == "hybrid" and scenario_config.finetune_level != "frozen":
                    axis.axvline(
                        phase_one_epochs + 0.5,
                        color="black",
                        linestyle="--",
                        label="Awal fase 2",
                    )
                axis.grid(alpha=0.25)
                axis.legend()
        figure.tight_layout()
        figure.savefig(figure_directory / f"training_curves_{model_name}.png", dpi=300)
        display(figure)
        plt.close(figure)


def plot_test_metric_comparison(selected_scenario_ids):
    figure_directory = REPORT_DIR / "figures"
    figure_directory.mkdir(parents=True, exist_ok=True)
    results = pd.read_csv(REPORT_DIR / "test_results.csv")
    screening = pd.read_csv(SCREENING_PATH)
    scenario_info = screening[["scenario_id", "model"]].drop_duplicates("scenario_id")
    results = results.merge(scenario_info, on="scenario_id")
    results = results[results.scenario_id.isin(selected_scenario_ids)]
    metrics = ("Accuracy", "Precision", "Recall", "F1")
    summary = results.groupby(
        ["scenario_id", "model"], sort=False
    )[list(metrics)].agg(["mean", "std"])
    figure, axes = plt.subplots(2, 2, figsize=(14, 10))
    for axis, metric in zip(axes.flat, metrics):
        means = summary[(metric, "mean")]
        deviations = summary[(metric, "std")].fillna(0)
        scenario_ids = means.index.get_level_values("scenario_id")
        model_names = means.index.get_level_values("model")
        positions = np.arange(len(means))
        for position, mean, deviation, model_name in zip(
            positions, means, deviations, model_names
        ):
            axis.errorbar(
                position,
                mean,
                yerr=deviation,
                fmt="o",
                color=MODEL_COLORS[model_name],
                capsize=4,
            )
            axis.annotate(
                f"{mean:.4f} ± {deviation:.4f}",
                (position, mean),
                xytext=(0, 8),
                textcoords="offset points",
                ha="center",
                fontsize=8,
            )
        axis.set_xticks(positions, scenario_ids, rotation=35, ha="right")
        axis.set_xlabel("Skenario")
        axis.set_ylabel(metric)
        metric_titles = {
            "Accuracy": "Akurasi",
            "Precision": "Presisi",
            "Recall": "Recall",
            "F1": "F1-score",
        }
        axis.set_title(metric_titles[metric])
        axis.set_ylim(
            max(0, float((means - deviations).min()) - 0.02),
            min(1, float((means + deviations).max()) + 0.04),
        )
        axis.grid(axis="y", alpha=0.25)
    figure.suptitle("Perbandingan Metrik Data Uji (rata-rata ± simpangan baku 3 seed)")
    figure.tight_layout()
    figure.savefig(REPORT_DIR / "figures" / "test_metric_comparison.png", dpi=300)
    display(figure)
    plt.close(figure)


def plot_efficiency_comparison(selected_scenario_ids):
    figure_directory = REPORT_DIR / "figures"
    figure_directory.mkdir(parents=True, exist_ok=True)
    results = pd.read_csv(REPORT_DIR / "test_results.csv")
    screening = pd.read_csv(SCREENING_PATH)
    scenario_info = screening[["scenario_id", "model"]].drop_duplicates("scenario_id")
    results = results.merge(scenario_info, on="scenario_id")
    results = results[results.scenario_id.isin(selected_scenario_ids)]
    efficiency = results.groupby(["scenario_id", "model"], sort=False)[
        ["Params", "Size (MB)", "Latency (ms/img)"]
    ].mean().reset_index()
    efficiency["Params (millions)"] = efficiency.Params / 1_000_000
    metrics = (
        ("Params (millions)", "Jumlah Parameter (juta)"),
        ("Size (MB)", "Ukuran Model (MB)"),
        ("Latency (ms/img)", "Waktu Inferensi (ms/citra)"),
    )
    figure, axes = plt.subplots(1, 3, figsize=(18, 5))
    for axis, (column, title) in zip(axes, metrics):
        positions = np.arange(len(efficiency))
        values = efficiency[column].to_numpy()
        colors = [MODEL_COLORS[name] for name in efficiency.model]
        axis.scatter(positions, values, c=colors)
        for position, value in zip(positions, values):
            axis.annotate(
                f"{value:.2f}",
                (position, value),
                xytext=(0, 6),
                textcoords="offset points",
                ha="center",
                fontsize=8,
            )
        axis.set_xticks(positions, efficiency.scenario_id, rotation=35, ha="right")
        axis.set(xlabel="Skenario", ylabel=title, title=title)
        axis.grid(axis="y", alpha=0.25)
    figure.suptitle("Perbandingan Efisiensi Model")
    figure.tight_layout()
    figure.savefig(REPORT_DIR / "figures" / "efficiency_comparison.png", dpi=300)
    display(figure)
    plt.close(figure)


def plot_test_per_class_f1(selected_scenario_ids):
    figure_directory = REPORT_DIR / "figures"
    figure_directory.mkdir(parents=True, exist_ok=True)
    results = pd.read_csv(REPORT_DIR / "test_per_class.csv")
    results = results[results.scenario_id.isin(selected_scenario_ids)]
    results = results.groupby(
        ["scenario_id", "Class"], sort=False
    ).F1.mean().rename("F1").reset_index()
    screening = pd.read_csv(SCREENING_PATH)
    scenario_info = screening[["scenario_id", "model"]].drop_duplicates("scenario_id")
    results = results.merge(scenario_info, on="scenario_id")
    scenario_ids = results.scenario_id.drop_duplicates().tolist()
    class_positions = np.arange(len(CLASS_DISPLAY_NAMES))
    bar_width = 0.8 / len(scenario_ids)
    figure, axis = plt.subplots(figsize=(14, 6))
    for scenario_index, scenario_id in enumerate(scenario_ids):
        scenario_values = results[results.scenario_id == scenario_id].set_index("Class")
        values = scenario_values.reindex(CLASS_DISPLAY_NAMES).F1.to_numpy()
        model_name = scenario_values.model.iloc[0]
        positions = class_positions - 0.4 + bar_width / 2 + scenario_index * bar_width
        axis.bar(
            positions,
            values,
            width=bar_width,
            color=MODEL_COLORS[model_name],
            label=scenario_id,
        )
    axis.set_xticks(class_positions, CLASS_DISPLAY_NAMES)
    axis.set(xlabel="Kelas", ylabel="F1-score", title="F1-score per Kelas")
    axis.legend(title="Skenario")
    axis.grid(axis="y", alpha=0.25)
    figure.tight_layout()
    figure.savefig(REPORT_DIR / "figures" / "test_per_class_f1.png", dpi=300)
    display(figure)
    plt.close(figure)


def plot_test_confusion_heatmaps(selected_scenario_ids):
    figure_directory = REPORT_DIR / "figures"
    figure_directory.mkdir(parents=True, exist_ok=True)
    for scenario_id in selected_scenario_ids:
        matrix_path = REPORT_DIR / "test_confusion" / f"{scenario_id}.csv"
        if not matrix_path.exists():
            raise FileNotFoundError(f"Test confusion matrix not found for {scenario_id}.")
        confusion = pd.read_csv(matrix_path, index_col=0).to_numpy()
        row_totals = confusion.sum(axis=1, keepdims=True)
        percentages = np.divide(
            confusion * 100,
            row_totals,
            out=np.zeros_like(confusion, dtype=float),
            where=row_totals != 0,
        )
        figure, axes = plt.subplots(1, 2, figsize=(14, 6))
        for axis, values, title, number_format in (
            (axes[0], confusion, "Jumlah", "d"),
            (axes[1], percentages, "Persentase per Baris (%)", ".1f"),
        ):
            image = axis.imshow(values, cmap="Blues")
            figure.colorbar(image, ax=axis)
            axis.set(
                xticks=np.arange(NUM_CLASSES),
                yticks=np.arange(NUM_CLASSES),
                xticklabels=CLASS_DISPLAY_NAMES,
                yticklabels=CLASS_DISPLAY_NAMES,
                xlabel="Kelas Prediksi",
                ylabel="Kelas Sebenarnya",
                title=title,
            )
            threshold = float(values.max()) / 2
            for row_index in range(NUM_CLASSES):
                for column_index in range(NUM_CLASSES):
                    value = values[row_index, column_index]
                    axis.text(
                        column_index,
                        row_index,
                        format(value, number_format),
                        ha="center",
                        va="center",
                        color="white" if value > threshold else "black",
                    )
        figure.suptitle(
            f"Matriks Konfusi Data Uji: {scenario_id} "
            "(jumlah dijumlahkan dari 3 seed)"
        )
        figure.tight_layout()
        figure.savefig(REPORT_DIR / "figures" / f"test_confusion_{scenario_id}.png", dpi=300)
        display(figure)
        plt.close(figure)


def visualize_final_results(selected_scenario_ids):
    if not selected_scenario_ids:
        raise ValueError("Selected scenario IDs cannot be empty.")
    figure_directory = REPORT_DIR / "figures"
    figure_directory.mkdir(parents=True, exist_ok=True)
    plot_training_curves(selected_scenario_ids)
    plot_test_metric_comparison(selected_scenario_ids)
    plot_efficiency_comparison(selected_scenario_ids)
    plot_test_per_class_f1(selected_scenario_ids)
    plot_test_confusion_heatmaps(selected_scenario_ids)

In [ ]:
visualize_final_results(select_final_models().scenario_id.tolist())